# Regimes along an ordering

A single trend line can hide the structure of a series that moves in
regimes: one level and slope for a decade, a break, then a different
pair. The `ordered_partitions` pass fits a model-based recursive
partition (MOB; Zeileis, Hothorn & Hornik 2008) of each numeric
response along the table's ordering column — a date, datetime, or
integer-year axis, or a column named by `hints.time`. A finding is
`changes/ordered_partition`; it reports dated segments with a level
and a slope, not a forecast and not a cause.

## The node model and the split search

Every node of the tree fits one model by OLS:

```text
y ~ level + slope * t
```

A node can split two ways, and the candidates compete on the same
scale — the two-segment SSE of the node's rows:

- **Along the ordering.** Every breakpoint whose two sides each
  hold at least the segment floor (24 rows) is scored exactly, from
  prefix sums of the node's sufficient statistics.
- **Across a categorical dimension.** The dimension's levels are
  ordered by their within-level fitted level at the node's mean
  `t` (then slope, then name), and every prefix bipartition of
  that ordering is scored. A node in which any row has no level
  on the dimension is not split on that dimension, so both sides
  always partition the node's rows exactly.

## The stopping bars

A split is accepted only when it clears both of two deterministic
bars; no p-value is computed or reported. The relative SSE gain
against the node's own fit must reach 2%, and the split's F
statistic (2 and n − 4 degrees of freedom, split model against the
node's single fit) must reach 15. The F floor is what keeps the
pass silent on noise: the best of ~n candidate breakpoints on a
pure-noise series still buys a few percent of SSE by chance. In a
calibration over 2,000 pure-noise series of 609 points, the
largest noise F was 11.96 — an F floor of 10 admitted 3 of 2,000
series, a floor of 15 admitted none. Recursion also stops at
depth 3 and 8 leaves at standard (depth 4, 12 leaves at deep).

Rows are capped per response (20,000 at standard, 100,000 at
deep) by a systematic sample in `t` order — every k-th row — so
break dates stay interpretable; coverage states when a sample
binds, how many responses were screened, and how many dimensions
were offered. Brief excursions shorter than the segment floor
are out of scope by design: a segment model cannot price a
three-month spike, and the breaks and surprises passes own that
territory.

In [1]:
import datetime

import numpy as np
import polars as pl
import onceover

rng = np.random.default_rng(7)
months = pl.date_range(pl.date(2010, 1, 1), pl.date(2024, 12, 1), "1mo", eager=True)
after_break = np.array([d >= datetime.date(2018, 1, 1) for d in months])
frames = []
for region, base in [("north", 3.0), ("south", 6.0)]:
    t = np.arange(len(months), dtype=float)
    y = base + 0.004 * t + np.where(after_break, 2.5, 0.0)
    y = y + rng.normal(0, 0.15, len(months))
    frames.append(pl.DataFrame({"month": months, "region": region, "rate": y}))
df = pl.concat(frames).sort(["month", "region"])
report = onceover.profile(df, detectors=["ordered_partitions"])
finding = report.select(category="changes", type="ordered_partition")[0]
metrics = finding["evidence"][0]["metrics"]
print(finding["claim"])
splits = pl.DataFrame(metrics["splits"])
splits

rate follows 4 level+trend regimes along month: first split region split {north} against {south}, 3 splits in all, level+trend SSE reduced 99.3%


f_statistic,gain_share,left_levels,on,right_levels,at
f64,f64,list[str],str,list[str],str
999.806335,0.848872,"[""north""]","""region""","[""south""]",null
1943.976028,0.068819,null,"""month""",null,"""2018-01-01"""
1801.062847,0.075505,null,"""month""",null,"""2018-01-01"""


In [2]:
leaves = pl.DataFrame(
    [
        {
            "period": f"{leaf['period'][0]} .. {leaf['period'][1]}",
            "subset": "; ".join(leaf["subset"]) or "(all rows)",
            "n": leaf["n"],
            "level_at_start": round(leaf["level_at_period_start"], 2),
            "slope_per_year": round(leaf["slope_per_year"], 3),
        }
        for leaf in metrics["leaves"]
    ]
)
leaves

period,subset,n,level_at_start,slope_per_year
str,str,i64,f64,f64
"""2010-01-01 .. 2017-12-01""","""region in {north}""",96,2.93,0.06
"""2018-01-01 .. 2024-12-01""","""region in {north}""",84,5.84,0.05
"""2010-01-01 .. 2017-12-01""","""region in {south}""",96,6.01,0.043
"""2018-01-01 .. 2024-12-01""","""region in {south}""",84,8.89,0.049


In [3]:
import datetime

import altair as alt

breaks = pl.DataFrame({"month": [datetime.date(2018, 1, 1)]})
lines = (
    alt.Chart(df).mark_line().encode(x="month:T", y="rate:Q", color="region:N")
)
rule = alt.Chart(breaks).mark_rule(color="black", strokeDash=[4, 3]).encode(x="month:T")
(lines + rule).properties(title="Two regions, one shared break at 2018-01")

alt.LayerChart(...)

## Read the tree before the headline

The claim names the first split and the total gain; the evidence
carries the whole tree. Each split states what it divides on — a
break date on the ordering column, or a dimension with the two
level sets — with its F statistic and its `gain_share`, the share
of the response's total (root) SSE that this one split removed.
Gain shares sum to at most the finding's `relative_gain`. Each
leaf states its period, the dimension conditions on its path from
the root, its row count, the fitted level at its first period,
and its slope per year — everything needed to refit the leaf by
hand from the rows it names.

Quote regimes leaf by leaf. A pooled trend across a break is a
number about no period the data actually contains; where a
downstream model needs one series, include the break dates as
controls or fit per leaf instead.